In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
from google.colab import files

print("Please upload the 'HRDataset_filled.xlsx' file.")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

# After uploading, you can re-run the cell above to load the dataframe.

Please upload the 'HRDataset_filled.xlsx' file.


In [ ]:
df = pd.read_excel("HRDataset_filled.xlsx")

df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['DateofTermination'] = df['DateofTermination'].fillna('Still Employed')

In [ ]:
manager_mapping = df.groupby('ManagerName')['ManagerID'].first()

df['ManagerID'] = df.apply(
    lambda row: manager_mapping[row['ManagerName']]
    if pd.isnull(row['ManagerID'])
    else row['ManagerID'],
    axis=1
)

In [ ]:
df['Attrition'] = np.where(
    df['EmploymentStatus'] == 'Voluntarily Terminated',
    'Yes',
    'No'
)

In [ ]:
sns.countplot(x='Attrition', data=df)

plt.title("Employee Attrition")
plt.show()

In [ ]:
sns.countplot(x='Sex', data=df)

plt.title("Gender Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(
    y='Department',
    data=df,
    order=df['Department'].value_counts().index
)

plt.title("Employees by Department")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    df['Salary'],
    bins=20
)

plt.title("Salary Distribution")
plt.show()

In [ ]:
pd.crosstab(
    df['Department'],
    df['Attrition']
).plot(kind='bar')

plt.title("Department vs Attrition")
plt.show()

In [ ]:
sns.boxplot(
    x='Attrition',
    y='Salary',
    data=df
)

plt.show()

In [ ]:
df['Attrition'] = df['Attrition'].map({
    'No':0,
    'Yes':1
})

In [ ]:
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
X = df.drop('Attrition', axis=1)

y = df['Attrition']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:",accuracy)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
importance = pd.DataFrame({
    'Feature':X.columns,
    'Importance':model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(10))

In [ ]:
probabilities = model.predict_proba(X_test)

risk_score = probabilities[:,1]

In [ ]:
def risk_category(score):

    if score >= 0.75:
        return "High Risk"

    elif score >= 0.50:
        return "Medium Risk"

    else:
        return "Low Risk"

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))